<a href="https://colab.research.google.com/github/rayvaril/student-dropout-capstone/blob/main/notebooks/05_explainability_fairness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Early Warning for Student Attrition

## An Explainable Machine Learning Approach with Fairness Evaluation

### 05 — Explainability and Fairness

This notebook evaluates the final early-warning models beyond predictive performance.

The goals are to:

- identify which features contribute most strongly to predicted dropout risk
- examine individual prediction explanations
- distinguish predictive signals from potentially actionable factors
- compare model performance across relevant student groups
- assess whether meaningful fairness disparities are present
- document limitations before any real-world use

In [52]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

In [53]:
repo_path = Path("/content/student-dropout-capstone")

if not repo_path.exists():
    !git clone https://github.com/rayvaril/student-dropout-capstone.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/student-dropout-capstone/notebooks

Repository already exists; skipping clone.
/content/student-dropout-capstone/notebooks


In [54]:
# Load final model artifacts and configuration

enrollment_model = joblib.load("../models/enrollment_logistic_model.joblib")
semester1_model = joblib.load("../models/semester1_logistic_model.joblib")

with open("../models/model_config.json", "r") as f:
    model_config = json.load(f)

enrollment_df = pd.read_csv("../data/processed/enrollment_model_base.csv")
semester1_df = pd.read_csv("../data/processed/semester1_model_base.csv")

print("Enrollment data:", enrollment_df.shape)
print("Semester 1 data:", semester1_df.shape)
print("\nModel configuration:")
print(model_config)

Enrollment data: (4424, 23)
Semester 1 data: (4424, 34)

Model configuration:
{'random_state': 42, 'test_size': 0.2, 'cross_validation_folds': 5, 'primary_metric': 'Dropout Recall', 'enrollment_model': {'algorithm': 'Logistic Regression', 'C': 0.1, 'class_weight': 'balanced'}, 'semester1_model': {'algorithm': 'Logistic Regression', 'C': 1, 'class_weight': 'balanced'}}


In [55]:
# Recreate the original held-out test split used in Notebook 04

target_col = "Dropout_binary"

y = enrollment_df[target_col]

all_indices = np.arange(len(enrollment_df))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=model_config["test_size"],
    random_state=model_config["random_state"],
    stratify=y
)

enrollment_test = enrollment_df.iloc[test_idx].copy()
semester1_test = semester1_df.iloc[test_idx].copy()

print("Training students:", len(train_idx))
print("Test students:", len(test_idx))
print("\nTest target distribution:")
print(enrollment_test[target_col].value_counts(normalize=True).sort_index())

Training students: 3539
Test students: 885

Test target distribution:
Dropout_binary
0    0.679096
1    0.320904
Name: proportion, dtype: float64


In [56]:
# Verify saved models on the recreated held-out test set

X_enrollment_test = enrollment_test.drop(columns=[target_col])
X_semester1_test = semester1_test.drop(columns=[target_col])

y_test = enrollment_test[target_col]

enrollment_pred = enrollment_model.predict(X_enrollment_test)
semester1_pred = semester1_model.predict(X_semester1_test)

verification_results = pd.DataFrame({
    "Model Stage": ["Enrollment", "Semester 1"],
    "Recall": [
        recall_score(y_test, enrollment_pred),
        recall_score(y_test, semester1_pred)
    ],
    "Precision": [
        precision_score(y_test, enrollment_pred),
        precision_score(y_test, semester1_pred)
    ],
    "F1": [
        f1_score(y_test, enrollment_pred),
        f1_score(y_test, semester1_pred)
    ]
})

verification_results.round(3)

,Model Stage,Recall,Precision,F1
0,Enrollment,0.722,0.526,0.608
1,Semester 1,0.775,0.698,0.735


In [57]:
print(enrollment_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=30,
                                                                sparse_output=False),
                                                  ['Marital status',
                                                   'Application mode', 'Course',
                                                   'Daytime/evening attendance',
                                                   'Previous qualification',
                                                   'Nacionality',
                                                   "Mother's qualification",
                                                   "Father's qualification",
                                                   "Mother's occupation",
                              

In [58]:
# Extract transformed feature names and Logistic Regression coefficients

enrollment_preprocessor = enrollment_model.named_steps["preprocessor"]
enrollment_classifier = enrollment_model.named_steps["classifier"]

enrollment_feature_names = enrollment_preprocessor.get_feature_names_out()
enrollment_coefficients = enrollment_classifier.coef_[0]

enrollment_coef_df = pd.DataFrame({
    "feature": enrollment_feature_names,
    "coefficient": enrollment_coefficients
})

enrollment_coef_df["absolute_coefficient"] = (
    enrollment_coef_df["coefficient"].abs()
)

enrollment_coef_df = enrollment_coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

enrollment_coef_df.head(20)

,feature,coefficient,absolute_coefficient
24,Course_9500,-0.835956,0.835956
48,Mother's qualification_34,0.671377,0.671377
20,Course_9130,0.651947,0.651947
8,Application mode_39,0.587565,0.587565
62,Mother's occupation_0,0.587416,0.587416
94,Scholarship holder_1,-0.567241,0.567241
93,Scholarship holder_0,0.560508,0.560508
28,Course_9853,0.530549,0.530549
73,Mother's occupation_infrequent_sklearn,-0.503884,0.503884
13,Application mode_infrequent_sklearn,-0.502964,0.502964


In [59]:
# Extract transformed feature names and coefficients for Semester 1 model

semester1_preprocessor = semester1_model.named_steps["preprocessor"]
semester1_classifier = semester1_model.named_steps["classifier"]

semester1_feature_names = semester1_preprocessor.get_feature_names_out()
semester1_coefficients = semester1_classifier.coef_[0]

semester1_coef_df = pd.DataFrame({
    "feature": semester1_feature_names,
    "coefficient": semester1_coefficients
})

semester1_coef_df["absolute_coefficient"] = (
    semester1_coef_df["coefficient"].abs()
)

semester1_coef_df = semester1_coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

semester1_coef_df.head(20)

,feature,coefficient,absolute_coefficient
48,Mother's qualification_34,1.321395,1.321395
28,Course_9853,1.216195,1.216195
45,Mother's qualification_4,-1.203756,1.203756
73,Mother's occupation_infrequent_sklearn,-1.131324,1.131324
85,Father's occupation_90,1.085487,1.085487
20,Course_9130,1.082192,1.082192
113,sem1_no_approved_units,-0.953600,0.953600
107,Curricular units 1st sem (approved),-0.877706,0.877706
86,Father's occupation_infrequent_sklearn,-0.847442,0.847442
14,Course_171,-0.837406,0.837406


In [35]:
# Enrollment model: permutation importance on the held-out test set
# Importance is measured by the change in Dropout Recall when each raw feature is shuffled.

enrollment_perm = permutation_importance(
    enrollment_model,
    X_enrollment_test,
    y_test,
    scoring="recall",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

enrollment_importance = pd.DataFrame({
    "feature": X_enrollment_test.columns,
    "importance_mean": enrollment_perm.importances_mean,
    "importance_std": enrollment_perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

enrollment_importance.head(15).round(3)

,feature,importance_mean,importance_std
16,Scholarship holder,0.059,0.014
3,Course,0.056,0.014
1,Application mode,0.036,0.016
15,Gender,0.028,0.011
12,Admission grade,0.027,0.013
17,Age at enrollment,0.027,0.010
19,Unemployment rate,0.015,0.011
8,Mother's qualification,0.015,0.009
9,Father's qualification,0.008,0.008
4,Daytime/evening attendance,0.008,0.007


In [36]:
# Semester 1 model: permutation importance on the held-out test set
# Importance is measured by the change in Dropout Recall when each raw feature is shuffled.

semester1_perm = permutation_importance(
    semester1_model,
    X_semester1_test,
    y_test,
    scoring="recall",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

semester1_importance = pd.DataFrame({
    "feature": X_semester1_test.columns,
    "importance_mean": semester1_perm.importances_mean,
    "importance_std": semester1_perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

semester1_importance.head(15).round(3)

,feature,importance_mean,importance_std
25,Curricular units 1st sem (approved),0.043,0.016
26,Curricular units 1st sem (grade),0.036,0.012
23,Curricular units 1st sem (enrolled),0.024,0.011
28,sem1_approval_rate,0.021,0.009
31,sem1_no_approved_units,0.020,0.011
30,sem1_completion_gap,0.018,0.012
19,Unemployment rate,0.017,0.009
22,Curricular units 1st sem (credited),0.016,0.007
16,Scholarship holder,0.010,0.008
29,sem1_evaluations_per_enrolled,0.010,0.009


### **Permutation Importance Assessment**

Permutation importance measures how much Dropout Recall decreases when a feature is randomly shuffled while the other features remain unchanged. Higher values indicate that the model relies more heavily on that feature for identifying students labeled as Dropout. These values describe predictive reliance and should not be interpreted as causal effects.

For the Enrollment model, the strongest signals include scholarship status, course, application mode, gender, admission grade, and age at enrollment. Because several demographic or socioeconomic variables appear among the stronger predictors, subgroup performance should be examined carefully during the fairness analysis.

After Semester 1 information becomes available, the importance pattern changes substantially. Academic-progress variables dominate the Semester 1 model, particularly approved units, Semester 1 grade, enrolled units, approval rate, and completion-related features.

This suggests that the later model relies more strongly on observed academic performance, while the earlier Enrollment model must depend more heavily on background and enrollment-stage characteristics. This provides an additional explanation for why the Semester 1 model achieves stronger predictive performance.

In [37]:
# Fairness evaluation by Gender on the held-out test set

def subgroup_metrics(df, y_true, y_pred, group_col, group_labels=None):
    temp = pd.DataFrame({
        group_col: df[group_col].to_numpy(),
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred)
    })

    rows = []

    for group_value, group_df in temp.groupby(group_col):
        label = (
            group_labels.get(group_value, group_value)
            if group_labels
            else group_value
        )

        tn, fp, fn, tp = confusion_matrix(
            group_df["y_true"],
            group_df["y_pred"],
            labels=[0, 1]
        ).ravel()

        rows.append({
            "group": label,
            "students": len(group_df),
            "actual_dropouts": int(group_df["y_true"].sum()),
            "recall": recall_score(
                group_df["y_true"],
                group_df["y_pred"],
                zero_division=0
            ),
            "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
            "precision": precision_score(
                group_df["y_true"],
                group_df["y_pred"],
                zero_division=0
            ),
            "selection_rate": group_df["y_pred"].mean()
        })

    return pd.DataFrame(rows)


gender_labels = {
    0: "Female",
    1: "Male"
}

enrollment_gender = subgroup_metrics(
    enrollment_test,
    y_test,
    enrollment_pred,
    "Gender",
    gender_labels
)

semester1_gender = subgroup_metrics(
    semester1_test,
    y_test,
    semester1_pred,
    "Gender",
    gender_labels
)

enrollment_gender.insert(0, "model_stage", "Enrollment")
semester1_gender.insert(0, "model_stage", "Semester 1")

gender_fairness = pd.concat(
    [enrollment_gender, semester1_gender],
    ignore_index=True
)

gender_fairness.round(3)

,model_stage,group,students,actual_dropouts,recall,false_negative_rate,precision,selection_rate
0,Enrollment,Female,564,138,0.572,0.428,0.470,0.298
1,Enrollment,Male,321,146,0.863,0.137,0.568,0.692
2,Semester 1,Female,564,138,0.725,0.275,0.649,0.273
3,Semester 1,Male,321,146,0.822,0.178,0.745,0.502


In [38]:
# Quantify gender performance gaps

gender_gap_rows = []

for stage, stage_df in gender_fairness.groupby("model_stage"):
    female = stage_df.loc[stage_df["group"] == "Female"].iloc[0]
    male = stage_df.loc[stage_df["group"] == "Male"].iloc[0]

    gender_gap_rows.append({
        "model_stage": stage,
        "recall_gap": abs(male["recall"] - female["recall"]),
        "false_negative_rate_gap": abs(
            male["false_negative_rate"] - female["false_negative_rate"]
        ),
        "precision_gap": abs(male["precision"] - female["precision"]),
        "selection_rate_gap": abs(
            male["selection_rate"] - female["selection_rate"]
        )
    })

gender_gaps = pd.DataFrame(gender_gap_rows)

gender_gaps.round(3)

,model_stage,recall_gap,false_negative_rate_gap,precision_gap,selection_rate_gap
0,Enrollment,0.291,0.291,0.097,0.394
1,Semester 1,0.097,0.097,0.096,0.229


In [39]:
enrollment_test["Age at enrollment"].describe()

,Age at enrollment
count,885.000000
mean,23.553672
std,8.211893
min,18.000000
25%,18.000000
50%,20.000000
75%,25.000000
max,70.000000


In [40]:
age_group = pd.cut(
    enrollment_test["Age at enrollment"],
    bins=[17, 20, 25, np.inf],
    labels=["18–20", "21–25", "26+"]
)

age_group_check = pd.DataFrame({
    "age_group": age_group,
    "Dropout_binary": y_test.to_numpy()
}).groupby(
    "age_group",
    observed=False
).agg(
    students=("Dropout_binary", "size"),
    actual_dropouts=("Dropout_binary", "sum")
)

age_group_check

,students,actual_dropouts
age_group,,
18–20,513,102
21–25,157,56
26+,215,126


In [41]:
# Fairness evaluation by Age Group on the held-out test set

age_groups = pd.cut(
    enrollment_test["Age at enrollment"],
    bins=[17, 20, 25, np.inf],
    labels=["18–20", "21–25", "26+"]
)

enrollment_test_age = enrollment_test.copy()
semester1_test_age = semester1_test.copy()

enrollment_test_age["Age group"] = age_groups.to_numpy()
semester1_test_age["Age group"] = age_groups.to_numpy()

enrollment_age = subgroup_metrics(
    enrollment_test_age,
    y_test,
    enrollment_pred,
    "Age group"
)

semester1_age = subgroup_metrics(
    semester1_test_age,
    y_test,
    semester1_pred,
    "Age group"
)

enrollment_age.insert(0, "model_stage", "Enrollment")
semester1_age.insert(0, "model_stage", "Semester 1")

age_fairness = pd.concat(
    [enrollment_age, semester1_age],
    ignore_index=True
)

age_fairness.round(3)

,model_stage,group,students,actual_dropouts,recall,false_negative_rate,precision,selection_rate
0,Enrollment,18–20,513,102,0.451,0.549,0.362,0.248
1,Enrollment,21–25,157,56,0.804,0.196,0.542,0.529
2,Enrollment,26+,215,126,0.905,0.095,0.633,0.837
3,Semester 1,18–20,513,102,0.627,0.373,0.577,0.216
4,Semester 1,21–25,157,56,0.750,0.250,0.627,0.427
5,Semester 1,26+,215,126,0.905,0.095,0.832,0.637


In [42]:
# Quantify age-group performance gaps

age_gap_rows = []

for stage, stage_df in age_fairness.groupby("model_stage"):
    age_gap_rows.append({
        "model_stage": stage,
        "recall_gap": stage_df["recall"].max() - stage_df["recall"].min(),
        "false_negative_rate_gap":
            stage_df["false_negative_rate"].max()
            - stage_df["false_negative_rate"].min(),
        "precision_gap":
            stage_df["precision"].max()
            - stage_df["precision"].min(),
        "selection_rate_gap":
            stage_df["selection_rate"].max()
            - stage_df["selection_rate"].min()
    })

age_gaps = pd.DataFrame(age_gap_rows)

age_gaps.round(3)

,model_stage,recall_gap,false_negative_rate_gap,precision_gap,selection_rate_gap
0,Enrollment,0.454,0.454,0.271,0.590
1,Semester 1,0.277,0.277,0.256,0.421


### **Gender and Age Fairness Assessment**

Subgroup evaluation on the held-out test set shows meaningful differences in model performance across both gender and age groups.

For gender, the Enrollment model has a Dropout Recall gap of 0.291 between female and male students. The gap decreases to 0.097 in the Semester 1 model. This indicates that adding first-semester academic information improves overall predictive performance while also reducing the gender difference in identifying students labeled as Dropout.

Age differences are larger. In the Enrollment model, recall ranges from 0.451 for students aged 18–20 to 0.905 for students aged 26+, producing a recall gap of 0.454. The Semester 1 model improves recall for the youngest group to 0.627, reducing the overall age-group recall gap to 0.277.

Selection-rate differences are also substantial across both gender and age groups, particularly in the Enrollment model.

These results identify subgroup performance disparities that require careful interpretation. They do not, by themselves, establish that the model is unfair because underlying dropout prevalence and student characteristics differ across groups. However, the size of the observed recall and false-negative-rate gaps means that the models should not be deployed without further fairness review, monitoring, and consideration of mitigation strategies.

In [43]:
scholarship_check = enrollment_test.groupby(
    "Scholarship holder"
)["Dropout_binary"].agg(
    students="size",
    actual_dropouts="sum"
)

scholarship_check

,students,actual_dropouts
Scholarship holder,,
0,654,255
1,231,29


In [44]:
scholarship_labels = {
    0: "No Scholarship",
    1: "Scholarship"
}

enrollment_scholarship = subgroup_metrics(
    enrollment_test,
    y_test,
    enrollment_pred,
    "Scholarship holder",
    scholarship_labels
)

semester1_scholarship = subgroup_metrics(
    semester1_test,
    y_test,
    semester1_pred,
    "Scholarship holder",
    scholarship_labels
)

enrollment_scholarship.insert(0, "model_stage", "Enrollment")
semester1_scholarship.insert(0, "model_stage", "Semester 1")

scholarship_fairness = pd.concat(
    [enrollment_scholarship, semester1_scholarship],
    ignore_index=True
)

scholarship_fairness.round(3)

,model_stage,group,students,actual_dropouts,recall,false_negative_rate,precision,selection_rate
0,Enrollment,No Scholarship,654,255,0.788,0.212,0.528,0.583
1,Enrollment,Scholarship,231,29,0.138,0.862,0.444,0.039
2,Semester 1,No Scholarship,654,255,0.816,0.184,0.715,0.445
3,Semester 1,Scholarship,231,29,0.414,0.586,0.500,0.104


In [45]:
# Quantify scholarship-status performance gaps

scholarship_gap_rows = []

for stage, stage_df in scholarship_fairness.groupby("model_stage"):
    no_scholarship = stage_df.loc[
        stage_df["group"] == "No Scholarship"
    ].iloc[0]

    scholarship = stage_df.loc[
        stage_df["group"] == "Scholarship"
    ].iloc[0]

    scholarship_gap_rows.append({
        "model_stage": stage,
        "recall_gap": abs(
            no_scholarship["recall"] - scholarship["recall"]
        ),
        "false_negative_rate_gap": abs(
            no_scholarship["false_negative_rate"]
            - scholarship["false_negative_rate"]
        ),
        "precision_gap": abs(
            no_scholarship["precision"] - scholarship["precision"]
        ),
        "selection_rate_gap": abs(
            no_scholarship["selection_rate"]
            - scholarship["selection_rate"]
        )
    })

scholarship_gaps = pd.DataFrame(scholarship_gap_rows)

scholarship_gaps.round(3)

,model_stage,recall_gap,false_negative_rate_gap,precision_gap,selection_rate_gap
0,Enrollment,0.650,0.650,0.083,0.544
1,Semester 1,0.402,0.402,0.215,0.341


### **Scholarship-Status Fairness Assessment**

Model performance also differs substantially by scholarship status.

In the Enrollment model, Dropout Recall is 0.788 for students without scholarships but only 0.138 for scholarship holders, producing a recall gap of 0.650. The Semester 1 model improves recall for scholarship holders to 0.414 and reduces the recall gap to 0.402, but a sizeable disparity remains.

Selection-rate differences are also large, although these must be interpreted alongside the substantially different underlying dropout prevalence between the two groups. Scholarship holders have a much lower observed dropout rate in the held-out test set than students without scholarships.

The scholarship-holder subgroup contains only 29 actual dropout cases in the test set, so these estimates are less stable than the gender and age comparisons. The results should therefore be treated as an important warning signal rather than definitive evidence of unfairness.

Overall, Semester 1 information reduces the disparity in identifying actual dropouts across scholarship groups, but further investigation would be required before deployment.

In [46]:
# Formal fairness-gap summary
# Demographic parity difference = gap in selection rates
# Equal opportunity difference = gap in recall / true-positive rates

fairness_summary = pd.DataFrame({
    "subgroup": [
        "Gender",
        "Gender",
        "Age group",
        "Age group",
        "Scholarship status",
        "Scholarship status"
    ],
    "model_stage": [
        "Enrollment",
        "Semester 1",
        "Enrollment",
        "Semester 1",
        "Enrollment",
        "Semester 1"
    ],
    "demographic_parity_difference": [
        gender_gaps.loc[
            gender_gaps["model_stage"] == "Enrollment",
            "selection_rate_gap"
        ].iloc[0],
        gender_gaps.loc[
            gender_gaps["model_stage"] == "Semester 1",
            "selection_rate_gap"
        ].iloc[0],
        age_gaps.loc[
            age_gaps["model_stage"] == "Enrollment",
            "selection_rate_gap"
        ].iloc[0],
        age_gaps.loc[
            age_gaps["model_stage"] == "Semester 1",
            "selection_rate_gap"
        ].iloc[0],
        scholarship_gaps.loc[
            scholarship_gaps["model_stage"] == "Enrollment",
            "selection_rate_gap"
        ].iloc[0],
        scholarship_gaps.loc[
            scholarship_gaps["model_stage"] == "Semester 1",
            "selection_rate_gap"
        ].iloc[0]
    ],
    "equal_opportunity_difference": [
        gender_gaps.loc[
            gender_gaps["model_stage"] == "Enrollment",
            "recall_gap"
        ].iloc[0],
        gender_gaps.loc[
            gender_gaps["model_stage"] == "Semester 1",
            "recall_gap"
        ].iloc[0],
        age_gaps.loc[
            age_gaps["model_stage"] == "Enrollment",
            "recall_gap"
        ].iloc[0],
        age_gaps.loc[
            age_gaps["model_stage"] == "Semester 1",
            "recall_gap"
        ].iloc[0],
        scholarship_gaps.loc[
            scholarship_gaps["model_stage"] == "Enrollment",
            "recall_gap"
        ].iloc[0],
        scholarship_gaps.loc[
            scholarship_gaps["model_stage"] == "Semester 1",
            "recall_gap"
        ].iloc[0]
    ]
})

fairness_summary.round(3)

,subgroup,model_stage,demographic_parity_difference,equal_opportunity_difference
0,Gender,Enrollment,0.394,0.291
1,Gender,Semester 1,0.229,0.097
2,Age group,Enrollment,0.590,0.454
3,Age group,Semester 1,0.421,0.277
4,Scholarship status,Enrollment,0.544,0.650
5,Scholarship status,Semester 1,0.341,0.402


### **Formal Fairness Gap Summary**

The table summarizes two fairness-related performance gaps on the held-out test set:

- **Demographic parity difference:** the absolute difference in model selection rates between groups. For age, this is calculated as the maximum minus minimum selection rate across the three age groups.
- **Equal opportunity difference:** the absolute difference in Dropout Recall between groups. For age, this is calculated as the maximum minus minimum recall across the three age groups.

Smaller values indicate more similar model behavior across the evaluated groups. A value of zero would indicate no observed gap for that metric in this test sample.

Across gender, age, and scholarship status, the Semester 1 model shows smaller demographic-parity and equal-opportunity gaps than the Enrollment model. However, meaningful disparities remain, particularly across age groups and scholarship status.

These metrics should not be interpreted as proof that a model is fair or unfair. They describe observed subgroup differences in this held-out sample and should be considered together with group sizes, underlying dropout prevalence, model purpose, and the consequences of false negatives and false positives.

In [47]:
# Add predicted probabilities to the held-out test set

enrollment_test_explain = enrollment_test.copy()
semester1_test_explain = semester1_test.copy()

enrollment_test_explain["predicted_dropout_probability"] = (
    enrollment_model.predict_proba(X_enrollment_test)[:, 1]
)

semester1_test_explain["predicted_dropout_probability"] = (
    semester1_model.predict_proba(X_semester1_test)[:, 1]
)

semester1_test_explain[
    [
        "Dropout_binary",
        "predicted_dropout_probability"
    ]
].sort_values(
    "predicted_dropout_probability",
    ascending=False
).head(10).round(3)

,Dropout_binary,predicted_dropout_probability
716,1,1.000
3533,1,0.999
2004,1,0.999
2447,0,0.999
477,1,0.998
1541,1,0.997
2930,1,0.997
916,1,0.996
734,1,0.995
2021,1,0.995


In [48]:
# Explain one high-risk Semester 1 prediction using Logistic Regression contributions

student_idx = 716

student_X = X_semester1_test.loc[[student_idx]]

student_transformed = semester1_preprocessor.transform(student_X)

student_contributions = pd.DataFrame({
    "feature": semester1_feature_names,
    "transformed_value": student_transformed[0],
    "coefficient": semester1_coefficients
})

student_contributions["contribution"] = (
    student_contributions["transformed_value"]
    * student_contributions["coefficient"]
)

student_contributions["absolute_contribution"] = (
    student_contributions["contribution"].abs()
)

student_contributions = student_contributions.sort_values(
    "absolute_contribution",
    ascending=False
)

print(
    "Actual outcome:",
    int(semester1_test.loc[student_idx, "Dropout_binary"])
)

print(
    "Predicted dropout probability:",
    round(
        semester1_test_explain.loc[
            student_idx,
            "predicted_dropout_probability"
        ],
        3
    )
)

student_contributions[
    ["feature", "contribution"]
].head(15).round(3)

Actual outcome: 1
Predicted dropout probability: 1.0


,feature,contribution
113,sem1_no_approved_units,-2.143
108,Curricular units 1st sem (grade),1.763
107,Curricular units 1st sem (approved),1.343
48,Mother's qualification_34,1.321
112,sem1_completion_gap,1.306
85,Father's occupation_90,1.085
20,Course_9130,1.082
110,sem1_approval_rate,0.797
62,Mother's occupation_0,0.685
39,Previous qualification_infrequent_sklearn,0.638


### **Individual Prediction Explanation**

The Semester 1 Logistic Regression model can also be decomposed at the individual-student level. Each transformed feature contributes to the model's predicted log-odds of dropout.

For the example student above, the model assigns a very high predicted dropout probability. Several Semester 1 academic-progress variables contribute strongly to this prediction, alongside selected enrollment-stage characteristics.

A positive contribution pushes the prediction toward the Dropout class, while a negative contribution pushes it away from Dropout.

These local contributions should be interpreted cautiously. Several academic features are correlated, so individual coefficient directions may appear counterintuitive when considered in isolation. The explanation describes how the fitted model produced the prediction; it does not establish that any feature caused the student's outcome.

Individual explanations should therefore be used as decision-support context rather than as automatic prescriptions for intervention.

In [49]:
# Explain a high-confidence false positive

student_idx = 2447

student_X = X_semester1_test.loc[[student_idx]]

student_transformed = semester1_preprocessor.transform(student_X)

false_positive_contributions = pd.DataFrame({
    "feature": semester1_feature_names,
    "transformed_value": student_transformed[0],
    "coefficient": semester1_coefficients
})

false_positive_contributions["contribution"] = (
    false_positive_contributions["transformed_value"]
    * false_positive_contributions["coefficient"]
)

false_positive_contributions["absolute_contribution"] = (
    false_positive_contributions["contribution"].abs()
)

false_positive_contributions = false_positive_contributions.sort_values(
    "absolute_contribution",
    ascending=False
)

print(
    "Actual outcome:",
    int(semester1_test.loc[student_idx, "Dropout_binary"])
)

print(
    "Predicted dropout probability:",
    round(
        semester1_test_explain.loc[
            student_idx,
            "predicted_dropout_probability"
        ],
        3
    )
)

false_positive_contributions[
    ["feature", "contribution"]
].head(15).round(3)

Actual outcome: 0
Predicted dropout probability: 0.999


,feature,contribution
112,sem1_completion_gap,2.825
104,Curricular units 1st sem (credited),1.837
100,Age at enrollment,1.242
105,Curricular units 1st sem (enrolled),-0.956
109,Curricular units 1st sem (without evaluations),-0.915
107,Curricular units 1st sem (approved),0.769
108,Curricular units 1st sem (grade),-0.728
101,Unemployment rate,0.592
110,sem1_approval_rate,0.589
23,Course_9254,0.542


### **False-Positive Case Assessment**

The model can also produce highly confident incorrect predictions.

The example above was assigned a very high predicted dropout probability even though the student's observed binary outcome was 0, meaning the student was not labeled as Dropout in the source dataset.

The prediction was driven by a combination of Semester 1 academic-progress indicators, age, economic context, and enrollment-stage characteristics. Individually, several of these signals resembled patterns associated with dropout in the training data, but their combination did not correspond to the student's observed outcome.

This illustrates an important limitation of individual risk scores: a high predicted probability is not a certainty. Even highly confident predictions can produce false positives.

For this reason, model outputs should be treated as signals for further review and supportive outreach rather than as automatic decisions about a student's future.

In [50]:
# Equalized-odds review
# Equalized odds considers both:
# 1. True-positive-rate (Recall) differences
# 2. False-positive-rate differences

def subgroup_equalized_odds(df, y_true, y_pred, group_col, group_labels=None):
    temp = pd.DataFrame({
        group_col: df[group_col].to_numpy(),
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred)
    })

    rows = []

    for group_value, group_df in temp.groupby(group_col):
        label = (
            group_labels.get(group_value, group_value)
            if group_labels
            else group_value
        )

        tn, fp, fn, tp = confusion_matrix(
            group_df["y_true"],
            group_df["y_pred"],
            labels=[0, 1]
        ).ravel()

        rows.append({
            "group": label,
            "recall": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
            "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan
        })

    result = pd.DataFrame(rows)

    recall_gap = result["recall"].max() - result["recall"].min()
    fpr_gap = (
        result["false_positive_rate"].max()
        - result["false_positive_rate"].min()
    )

    return recall_gap, fpr_gap, max(recall_gap, fpr_gap)


equalized_odds_rows = []

for subgroup_name, enroll_df, sem1_df, group_col, labels in [
    (
        "Gender",
        enrollment_test,
        semester1_test,
        "Gender",
        gender_labels
    ),
    (
        "Age group",
        enrollment_test_age,
        semester1_test_age,
        "Age group",
        None
    ),
    (
        "Scholarship status",
        enrollment_test,
        semester1_test,
        "Scholarship holder",
        scholarship_labels
    )
]:
    for stage, df_stage, preds in [
        ("Enrollment", enroll_df, enrollment_pred),
        ("Semester 1", sem1_df, semester1_pred)
    ]:
        recall_gap, fpr_gap, eo_gap = subgroup_equalized_odds(
            df_stage,
            y_test,
            preds,
            group_col,
            labels
        )

        equalized_odds_rows.append({
            "subgroup": subgroup_name,
            "model_stage": stage,
            "recall_gap": recall_gap,
            "false_positive_rate_gap": fpr_gap,
            "equalized_odds_gap": eo_gap
        })

equalized_odds_summary = pd.DataFrame(equalized_odds_rows)

equalized_odds_summary.round(3)

,subgroup,model_stage,recall_gap,false_positive_rate_gap,equalized_odds_gap
0,Gender,Enrollment,0.291,0.340,0.340
1,Gender,Semester 1,0.097,0.108,0.108
2,Age group,Enrollment,0.454,0.544,0.544
3,Age group,Semester 1,0.277,0.144,0.277
4,Scholarship status,Enrollment,0.650,0.426,0.650
5,Scholarship status,Semester 1,0.402,0.149,0.402


### **Equalized-Odds Assessment**

Equalized odds evaluates whether model error behavior is comparable across groups by considering both true-positive-rate differences and false-positive-rate differences.

For this analysis, the equalized-odds gap is summarized as the larger of the subgroup Recall gap and false-positive-rate gap. Smaller values indicate more similar error behavior across the evaluated groups.

The Semester 1 model shows smaller equalized-odds gaps than the Enrollment model across all three subgroup dimensions:

- Gender: 0.340 to 0.108
- Age group: 0.544 to 0.277
- Scholarship status: 0.650 to 0.402

This reinforces the earlier finding that adding Semester 1 academic information reduces several observed subgroup disparities. However, the remaining gaps—particularly for age and scholarship status—are still large enough to warrant caution.

These results describe the held-out test sample only and should not be interpreted as proof of fairness or unfairness in future student populations.

In [51]:
# Save explainability and fairness results

results_path = Path("../results")
results_path.mkdir(exist_ok=True)

enrollment_importance.to_csv(
    results_path / "enrollment_permutation_importance.csv",
    index=False
)

semester1_importance.to_csv(
    results_path / "semester1_permutation_importance.csv",
    index=False
)

gender_fairness.to_csv(
    results_path / "gender_fairness_results.csv",
    index=False
)

age_fairness.to_csv(
    results_path / "age_fairness_results.csv",
    index=False
)

scholarship_fairness.to_csv(
    results_path / "scholarship_fairness_results.csv",
    index=False
)

fairness_summary.to_csv(
    results_path / "fairness_summary.csv",
    index=False
)

equalized_odds_summary.to_csv(
    results_path / "equalized_odds_summary.csv",
    index=False
)

print("Explainability and fairness results saved successfully.")

Explainability and fairness results saved successfully.


## **Final Explainability and Fairness Assessment**

The explainability analysis shows that the two early-warning stages rely on different types of information.

At Enrollment, the model depends more heavily on background, application, demographic, and socioeconomic characteristics such as scholarship status, course, application mode, gender, admission grade, and age.

After Semester 1, the model shifts toward observed academic-progress signals. Approved units, Semester 1 grade, enrolled units, approval rate, and completion-related features become the strongest contributors to Dropout Recall.

The fairness analysis identifies meaningful subgroup performance differences. The Enrollment model shows particularly large gaps across gender, age, and scholarship status. Adding Semester 1 academic information reduces the observed gaps across all three subgroup dimensions, but does not eliminate them.

The Semester 1 model therefore provides two advantages: stronger overall predictive performance and smaller observed subgroup disparities. However, notable fairness concerns remain, particularly across age groups and scholarship status.

These findings support using the models only as decision-support tools. Risk scores should guide further review and supportive outreach rather than automatically determine interventions or student outcomes. Before real-world deployment, the models would require validation on future cohorts, continued subgroup monitoring, and consideration of fairness-mitigation strategies.